In [1]:
import os, sys, sqlite3

In [2]:
DB_PATH = "experiment.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

setup = sqlite3.connect(DB_PATH, autocommit=False)
setup.execute("CREATE TABLE clients (id INTEGER PRIMARY KEY, name TEXT NOT NULL);")
setup.commit()
setup.close()

def name(conn):
    return [[r[0] for r in conn.execute("SELECT name FROM clients").fetchall()]]

In [7]:
c1 = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)
c2 = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)
print("c1 in_transaction: ", c1.in_transaction, " | c2 in_transaction: ", c2.in_transaction)

c1 in_transaction:  True  | c2 in_transaction:  True


In [8]:
c1.execute("INSERT INTO clients (name) VALUES (?)", ("Farm A", ))
print("c1 sees: ", name(c1))

c1 sees:  [['Farm A']]


In [9]:
print("c2 sees: ", name(c2))
print("c2 in_transaction: ", c2.in_transaction)

c2 sees:  [[]]
c2 in_transaction:  True


In [10]:
try:
    c1.commit()
    print("c1 commit succeeded")
except sqlite3.Error as err:
    print("c1 commit failed: ", type(err).__name__, "-", err)
    print("c1 in_transaction: ", c1.in_transaction)

c1 commit failed:  OperationalError - database is locked
c1 in_transaction:  True


In [11]:
c2.rollback()
print("c2 in_transaction after rollback: ", c2.in_transaction)

if c1.in_transaction:
    c1.commit()
    print("c1 commit retried and succeeded")

print("c2 sees now: ", name(c2))

c2 in_transaction after rollback:  True
c1 commit retried and succeeded
c2 sees now:  [['Farm A']]


In [12]:
c1.close()
c2.close()